In [1]:
# Import necessary modules
import sys
from pathlib import Path
import numpy as np
import pickle

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from database import db
from student import student_manager
from embedding_generator import embedding_generator

2026-07-03 12:43:07,210 - database - INFO - Successfully connected to MongoDB database: attendance_system
2026-07-03 12:43:10,136 - numexpr.utils - INFO - NumExpr defaulting to 12 threads.
2026-07-03 12:43:14,159 - keras_facenet.embedding_model - INFO - Loading weights.
2026-07-03 12:43:14,160 - keras_facenet.utils - INFO - Looking for C:\Users\sanja/.keras-facenet\20180402-114759\20180402-114759-weights.h5
2026-07-03 12:43:19,809 - tensorflow - WARNING - From C:\Users\sanja\AppData\Roaming\Python\Python313\site-packages\keras\src\backend\tensorflow\core.py:233: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

2026-07-03 12:43:20,674 - embedding_generator - INFO - FaceNet model loaded successfully


In [2]:
# Check embedding status
summary = embedding_generator.get_embedding_summary()

print("Embedding Status:")
print("-" * 40)
print(f"Total Students: {summary['total_students']}")
print(f"Face Captured: {summary['face_captured']}")
print(f"Embeddings Generated: {summary['embeddings_generated']}")
print(f"Pending: {summary['pending']}")

if summary['embeddings_generated'] < summary['face_captured']:
    print(f"\n⚠ {summary['pending']} students need embeddings")

Embedding Status:
----------------------------------------
Total Students: 5
Face Captured: 1
Embeddings Generated: 1
Pending: 0


In [3]:
# Generate embeddings for a specific student
roll_number = input("\nEnter roll number to generate embeddings: ").strip()

if roll_number:
    success, message, embedding = embedding_generator.generate_embeddings_for_student(roll_number)

    if success:
        print(f"\n✓ {message}")

        if embedding is not None:
            print(f"Embedding shape: {embedding.shape}")
            print(f"Embedding type: {type(embedding)}")
            print(f"First 5 values: {embedding[:5]}")
    else:
        print(f"\n✗ {message}")


Enter roll number to generate embeddings:  22002



✗ No face images found for Priya Sharma


In [4]:
# Generate embeddings for all pending students
choice = input("\nGenerate embeddings for all pending students? (y/n): ").strip().lower()

if choice == "y":
    results = embedding_generator.generate_all_embeddings()

    print("\nEmbedding Generation Results:")
    print("-" * 40)
    print(f"Successful: {results['successful']}")
    print(f"Failed: {results['failed']}")

    print("\nDetails:")
    for detail in results["details"]:
        print(f"  {detail['name']}: {detail['status']}")


Generate embeddings for all pending students? (y/n):  n


In [5]:
# Load and analyze embeddings
embeddings = embedding_generator.load_all_embeddings()

print(f"Loaded {len(embeddings)} embeddings")

if embeddings:

    print("\nEmbedding Analysis:")
    print("-" * 40)

    # Get dimensions
    first_embedding = next(iter(embeddings.values()))
    print(f"Embedding dimension: {len(first_embedding)}")

    # Calculate statistics
    all_embeddings = np.array(list(embeddings.values()))

    print(f"Mean: {np.mean(all_embeddings):.4f}")
    print(f"Std: {np.std(all_embeddings):.4f}")
    print(f"Min: {np.min(all_embeddings):.4f}")
    print(f"Max: {np.max(all_embeddings):.4f}")

    print("\nStudent Embeddings:")

    for roll, emb in embeddings.items():
        print(f"  {roll}: shape {emb.shape}")

2026-07-03 12:43:29,809 - embedding_generator - INFO - Loaded 1 embeddings from database


Loaded 1 embeddings

Embedding Analysis:
----------------------------------------
Embedding dimension: 512
Mean: 0.0028
Std: 0.0441
Min: -0.1222
Max: 0.1242

Student Embeddings:
  22001: shape (512,)


In [6]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


def visualize_embeddings(embeddings_dict):
    """Visualize embeddings using PCA."""

    if len(embeddings_dict) < 2:
        print("Need at least 2 embeddings for visualization")
        return

    # Prepare data
    rolls = list(embeddings_dict.keys())
    embeddings = np.array([embeddings_dict[roll] for roll in rolls])

    # PCA
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)

    # Plot
    plt.figure(figsize=(10, 8))
    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=100, alpha=0.7)

    # Labels
    for i, roll in enumerate(rolls):
        plt.annotate(
            roll,
            (embeddings_2d[i, 0], embeddings_2d[i, 1]),
            fontsize=8,
            alpha=0.7
        )

    plt.title("Face Embeddings Visualization (PCA)")
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


if len(embeddings) >= 2:
    visualize_embeddings(embeddings)
else:
    print("Need at least 2 embeddings for visualization")

Need at least 2 embeddings for visualization


In [7]:
from sklearn.metrics.pairwise import cosine_similarity


def check_similarity(embeddings_dict):
    """Check similarity between embeddings."""

    if len(embeddings_dict) < 2:
        print("Need at least 2 embeddings for similarity check")
        return

    rolls = list(embeddings_dict.keys())
    embeddings = np.array([embeddings_dict[roll] for roll in rolls])

    similarities = cosine_similarity(embeddings)

    print("Embedding Similarity Matrix:")
    print("-" * 50)

    print(f"{'':<10}", end="")
    for roll in rolls[:5]:
        print(f"{roll:<10}", end="")
    print()

    for i, roll in enumerate(rolls[:5]):
        print(f"{roll:<10}", end="")
        for j in range(min(5, len(rolls))):
            print(f"{similarities[i, j]:.3f}    ", end="")
        print()

    mask = ~np.eye(len(embeddings), dtype=bool)

    print("\nStatistics:")
    print(f"Average similarity: {np.mean(similarities[mask]):.3f}")
    print(f"Max similarity: {np.max(similarities[mask]):.3f}")
    print(f"Min similarity: {np.min(similarities[mask]):.3f}")


if len(embeddings) >= 2:
    check_similarity(embeddings)

In [8]:
# Save embeddings to file
embeddings_file = Path("../embeddings/embeddings.pkl")
embeddings_file.parent.mkdir(exist_ok=True)

with open(embeddings_file, "wb") as f:
    pickle.dump(embeddings, f)

print(f"✓ Embeddings saved to {embeddings_file}")
print(f"File size: {embeddings_file.stat().st_size / 1024:.2f} KB")

✓ Embeddings saved to ..\embeddings\embeddings.pkl
File size: 4.16 KB


In [9]:
# Close database connection
db.close()

print("\n✓ Database connection closed")

2026-07-03 12:43:30,553 - database - INFO - MongoDB connection closed



✓ Database connection closed
